# TGLF: extent vs WIDTH for each NBASIS section (fig6 of gftm_quadrature_extent, for TGLF)

Fusion_PhD-38e9.24. The GFTM figure `fig6_extent_vs_width_per_nbasis` (notebook `gftm_quadrature_extent.ipynb`, branch
`analysis/gftm-quadrature-extent`, code cell 18) repeated for TGLF over TGLF's accepted NBASIS range, at three FILTER values.

**TGLF's accepted window** (`gacode/tglf/src`, `tglf_inout.f90:286-292`, `put_switches`): NBASIS_MAX must lie in 2..32
(`nb = 32`, `tglf_modules.f90:8`) or it is RESET to the default 4; odd NBASIS_MAX is rounded down to even; NXGRID must
satisfy 2*NXGRID-1 <= nxm = 63 (NXGRID <= 32) or it is reset to 16. There is no NBASIS-vs-NXGRID relation in the source.

Sections: NBASIS_MAX 4, 8, 16, 24, 32, all at NXGRID 32 (TGLF's maximum), so only NBASIS changes between sections.
Leaves `TGLF/Runs/kbm_tglf_nbasis/<case>/nb<N>_nx32_<filter>_<width>` (Fusion_PhD registry family `tglf_nbasis_extent`): the
`tglf_width_oracle` decks (THETA_TRAPPED 0.7, FIND_WIDTH F, NMODES 4, 24 widths 0.2-4.0) with NBASIS_MAX, NXGRID and FILTER
changed. FILTER 0.0 (off), 2.0 (TGLF's default frequency filter), -0.1 (TGLF's parity gate).

Metric, as the GFTM figure: median over the GS2-KBM population (user classifier, `general_analysis.mode_classification`) of
|log10(Extent_TGLF / Extent_GS2)| for the dominant mode (argmax growth rate, growth > 0), pyro `Extent` at 95%.
Red dotted line: largest WIDTH, contiguous upward from the panel's most accurate one, with phi Med |log(Extent Ratio)| <=
log10(1.12). **No grey band**: GFTM's band came from a Gaussian test on GFTM's basis and nodes, and it has not been
recomputed for TGLF.

In [ ]:
from pathlib import Path
import os
import warnings
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import pyrokinetics
from pyrokinetics.pyroscan import PyroScanGKOutput
from pyrokinetics.diagnostics.extent import Extent
from general_analysis import mode_classification as mc

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)
print("pyrokinetics", pyrokinetics.__version__)

analysis_name = "tglf_quadrature_extent"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
project = "LATIN_HYPERCUBE"
cases = {"SPR-045": "SPR-045", "M1": "M1"}
fraction = 0.95                         # pyro Extent: theta range holding 95% of |field|^2
fields = ["phi", "apar"]
field_labels = {"phi": r"$\phi$", "apar": r"$A_\parallel$"}
sections = [4, 8, 16, 24, 32]           # NBASIS_MAX, all at NXGRID 32
filters = {"fp0p0": 0.0, "fp2p0": 2.0, "fm0p1": -0.1}
fs_title, fs_legend = 16, 12


def log_xticks(ax, ticks):
    lims = ax.get_xlim()
    ax.set_xticks(ticks)
    ax.set_xlim(lims)
    ax.xaxis.set_major_formatter(StrMethodFormatter("{x:g}"))
    ax.minorticks_off()


def mag(v):
    return np.asarray(getattr(v.data, "magnitude", v.data))


def extent(theta, power):
    return float(Extent._compute(xr.DataArray(power, dims="theta", coords={"theta": theta}), fraction)[0])

## GS2 reference

Eigenfunctions (final time) and the KBM label from the GS2 `pyro_cube`, growth rates from `pyro_cube_avg`, as the GFTM
notebook; the label uses the cube-based classifier (`mc.load_gs2_cube` + `indicators`), not a per-run load.

In [ ]:
warnings.filterwarnings("ignore")
gs2 = {}
for db, case in cases.items():
    base = data_root / "GS2" / "Runs" / project / case / "kbm_8d"
    scan, last = mc.load_gs2_cube(base, "pyro_cube")
    avg = PyroScanGKOutput.from_netcdf(base / "pyro_cube_avg" / "cube.nc").data
    names = avg.sample_name.values.astype(str)
    assert (last.sample_name.values.astype(str) == names).all()
    label = np.asarray(mc.classify(mc.indicators(scan, last)))
    ef = last.eigenfunctions.transpose("sample", "field", "theta")
    theta = mag(ef.theta)
    g = mag(avg.growth_rate)
    gs2[db] = dict(names=names, label=label, gamma=g, kbm=(label == "KBM") & np.isfinite(g) & (g > 0),
                   ext={f: np.array([extent(theta, np.abs(e) ** 2) for e in mag(ef.sel(field=f))]) for f in fields})
    print(db, pd.Series(label).value_counts().to_dict(), "| KBM population:", gs2[db]["kbm"].sum())

## TGLF leaves and the accuracy table

In [ ]:
def dominant(db, path):
    out = PyroScanGKOutput.from_netcdf(path)
    Extent(out, fraction=fraction)
    d = out.data
    idx = pd.Index(d.sample_name.values.astype(str)).get_indexer(gs2[db]["names"])
    assert (idx >= 0).all()
    gam = mag(d.growth_rate.transpose("sample", "mode"))[idx]
    g = np.where(gam > 0, gam, -np.inf)
    dom, has = g.argmax(1), np.isfinite(g.max(1))
    pick = lambda a: np.where(has, a[np.arange(len(a)), dom], np.nan)
    return dict(gamma=np.where(has, gam[np.arange(len(gam)), dom], 0.0),
                ext={f: pick(mag(d.extent.sel(field=f).transpose("sample", "mode"))[idx]) for f in fields})


tglf = {}
for db, case in cases.items():
    for ft in filters:
        for nb in sections:
            for leaf in (data_root / "TGLF" / "Runs" / "kbm_tglf_nbasis" / case).glob(f"nb{nb}_nx32_{ft}_w*"):
                W = float(leaf.name.split("_")[-1][1:].replace("p", "."))
                tglf[db, ft, nb, W] = dominant(db, leaf / "pyroscan.nc")
print("widths per (filter, section):", pd.Series([k[1:3] for k in tglf if k[0] == "SPR-045"]).value_counts().sort_index().to_dict())
acc = []
for (db, ft, nb, W), e in tglf.items():
    k = gs2[db]["kbm"]
    acc.append(dict(db=db, ft=ft, nb=nb, W=W, **{f"ext_{f}": np.nanmedian(np.abs(np.log10(e["ext"][f][k] / gs2[db]["ext"][f][k]))) for f in fields}))
acc = pd.DataFrame(acc).sort_values(["db", "ft", "nb", "W"])
acc["ok_ext"] = acc.ext_phi <= np.log10(1.12)


def last_ok(a, metric, ok):
    i = int(a[metric].idxmin())
    while i + 1 < len(a) and a[ok][i + 1]:
        i += 1
    return a.W[i] if a[ok][i] else np.nan


w_max_ext = {}
for db in cases:
    for ft in filters:
        for nb in sections:
            a = acc[(acc.db == db) & (acc.ft == ft) & (acc.nb == nb)].reset_index(drop=True)
            w_max_ext[db, ft, nb] = last_ok(a, "ext_phi", "ok_ext")
print(pd.Series(w_max_ext).unstack().round(3).to_string())

## fig6, one figure per FILTER

In [ ]:
def per_nbasis(cols, labels, ylabel, w_max, ft):
    fig, axes = plt.subplots(len(cases), len(sections), figsize=(22, 9), sharex=True, sharey=True, squeeze=False)
    for r, db in enumerate(cases):
        for j, nb in enumerate(sections):
            ax = axes[r, j]
            a = acc[(acc.db == db) & (acc.ft == ft) & (acc.nb == nb)]
            for col, lab, cc in zip(cols, labels, ("C0", "C3")):
                ax.plot(a.W, a[col], color=cc, label=lab)
            if np.isfinite(w_max[db, ft, nb]):
                ax.axvline(w_max[db, ft, nb], color="red", ls=":", lw=2, marker="")
            ax.set(xscale="log")
            ax.set_title(f"{db}, NBASIS {nb}", fontsize=fs_title)
            if r == len(cases) - 1:
                ax.set_xlabel("WIDTH")
                log_xticks(ax, [0.2, 0.5, 1, 2, 4])
        axes[r, 0].set_ylabel(ylabel)
    if len(cols) > 1:
        axes[0, 0].legend(fontsize=fs_legend)
    plt.show()
    return fig


fig6 = {ft: per_nbasis(["ext_phi", "ext_apar"], [field_labels["phi"], field_labels["apar"]], "Med |log(Extent Ratio)|", w_max_ext, ft)
        for ft in filters}

In [ ]:
out = ROOT / "Plots" / analysis_name
out.mkdir(parents=True, exist_ok=True)
for ft, fig in fig6.items():
    fig.savefig(out / f"fig6_tglf_extent_vs_width_per_nbasis_{ft.replace('fp', 'f')}.png", bbox_inches="tight")
acc.round(4).to_csv(out / "accuracy_vs_width.csv", index=False)     # medians per leaf only, no per-case rows
print(sorted(p.name for p in out.iterdir()))